# In Class Activity April 14th, 2026

### Importing libraries, preparing data, initial EDA

In [2]:
# importing libraries (feel free to add more if you want to explore other things)
import numpy as np
import pandas as pd
import sweetviz as sv
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from xgboost import XGBClassifier
from sklearn.metrics import f1_score, accuracy_score, classification_report
import optuna


In [3]:
# importing data
adult = pd.read_csv('adult.csv')
adult.head(20)

,age,workclass,fnlwgt,education,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,11th,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,<=50K
1,38,Private,89814,HS-grad,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,<=50K
2,28,Local-gov,336951,Assoc-acdm,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,>50K
3,44,Private,160323,Some-college,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,>50K
4,18,?,103497,Some-college,10,Never-married,?,Own-child,White,Female,0,0,30,United-States,<=50K
5,34,Private,198693,10th,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,<=50K
6,29,?,227026,HS-grad,9,Never-married,?,Unmarried,Black,Male,0,0,40,United-States,<=50K
7,63,Self-emp-not-inc,104626,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,>50K
8,24,Private,369667,Some-college,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,<=50K
9,55,Private,104996,7th-8th,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,<=50K


In [5]:
# initial EDA with sweetviz
report = sv.analyze(adult)
report.show_html('adult_sweet_report.html')

# you are welcome to replace this cell with your own EDA, but make sure to include
# some visualizations and insights about the data


                                             |          | [  0%]   00:00 -> (? left)

Report adult_sweet_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [9]:
### missing values
missing_values = adult.isnull().sum()
print("Missing values in each column:")
print(missing_values)
### value counts where value is ?
### only show counts for ? values
for column in adult.columns:
    if adult[column].dtype == 'object':  # Check if the column is categorical
        count_question_mark = (adult[column] == '?').sum()
        if count_question_mark > 0:
            print(f"Column '{column}' has {count_question_mark} entries with '?'")

Missing values in each column:
age                0
workclass          0
fnlwgt             0
education          0
educational-num    0
marital-status     0
occupation         0
relationship       0
race               0
gender             0
capital-gain       0
capital-loss       0
hours-per-week     0
native-country     0
income             0
dtype: int64
Column 'workclass' has 2799 entries with '?'
Column 'occupation' has 2809 entries with '?'
Column 'native-country' has 857 entries with '?'


### In the markdown cell below describe what you learned from your EDA and how it will inform your modeling decisions





The dataset is skewed, with majority of people making less than $50k. Additionally, majority of participants are native to the Unted States, and more men and white people were asked.

There is colinearity between education and educational-num indicating one needs to be removed. 

There is also likely colinearity between relationship and marital status. Another strong relationships include occupation and educational-num.

To address these challenges, as well as the variables represented with a ?, data preprocessing must occur. 

### Data Preprocessing (minimal) and Baseline Model

In [10]:
# data cleaning and preprocessing

# changing ? to NaN
adult = adult.replace('?', np.nan)

#education and education num are redundant, so we can drop one of them
adult = adult.drop('education', axis=1)

# target variable is income with 2 levels, so we can encode it as 0 and 1
adult['income'] = adult['income'].apply(lambda x: 1 if x == '>50K' else 0)

# change dtype categorical variables to category
categorical_cols = adult.select_dtypes(include='object').columns
adult[categorical_cols] = adult[categorical_cols].astype('category')


adult.head(20)

,age,workclass,fnlwgt,educational-num,marital-status,occupation,relationship,race,gender,capital-gain,capital-loss,hours-per-week,native-country,income
0,25,Private,226802,7,Never-married,Machine-op-inspct,Own-child,Black,Male,0,0,40,United-States,0
1,38,Private,89814,9,Married-civ-spouse,Farming-fishing,Husband,White,Male,0,0,50,United-States,0
2,28,Local-gov,336951,12,Married-civ-spouse,Protective-serv,Husband,White,Male,0,0,40,United-States,1
3,44,Private,160323,10,Married-civ-spouse,Machine-op-inspct,Husband,Black,Male,7688,0,40,United-States,1
4,18,NaN,103497,10,Never-married,NaN,Own-child,White,Female,0,0,30,United-States,0
5,34,Private,198693,6,Never-married,Other-service,Not-in-family,White,Male,0,0,30,United-States,0
6,29,NaN,227026,9,Never-married,NaN,Unmarried,Black,Male,0,0,40,United-States,0
7,63,Self-emp-not-inc,104626,15,Married-civ-spouse,Prof-specialty,Husband,White,Male,3103,0,32,United-States,1
8,24,Private,369667,10,Never-married,Other-service,Unmarried,White,Female,0,0,40,United-States,0
9,55,Private,104996,4,Married-civ-spouse,Craft-repair,Husband,White,Male,0,0,10,United-States,0


In [12]:
percent_missing = adult.isnull().mean() * 100
missing_values = adult.isnull().sum()
print("Missing values in each column:")
print(missing_values)
print("\nPercentage of missing values in each column:")
print(percent_missing)


Missing values in each column:
age                   0
workclass          2799
fnlwgt                0
educational-num       0
marital-status        0
occupation         2809
relationship          0
race                  0
gender                0
capital-gain          0
capital-loss          0
hours-per-week        0
native-country      857
income                0
dtype: int64

Percentage of missing values in each column:
age                0.000000
workclass          5.730724
fnlwgt             0.000000
educational-num    0.000000
marital-status     0.000000
occupation         5.751198
relationship       0.000000
race               0.000000
gender             0.000000
capital-gain       0.000000
capital-loss       0.000000
hours-per-week     0.000000
native-country     1.754637
income             0.000000
dtype: float64


In [13]:
### drop rows with missing values
adult = adult.dropna()

In [16]:
# defining features and target variable
X = adult.drop('income', axis=1)
y = adult['income']

# splitting data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, 
                                                    random_state=222, stratify=y)

In [17]:
# building xgboost default model and evaluating with stratified k-fold cross validation
xgb_cv = XGBClassifier(enable_categorical=True, random_state=222)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)
cv_scores = cross_val_score(xgb_cv, X, y, cv=skf, scoring='f1')
print(f'Cross-validated F1 scores: {cv_scores}')
print(f'Mean F1 score: {cv_scores.mean()}') 


Cross-validated F1 scores: [0.71353167 0.71694465 0.70278114 0.71486519 0.72525885]
Mean F1 score: 0.7146762977427369


### Use the markdown cell below to describe your baseline model performance and how you will try to improve it

The baseline model performance has a mean F1 score of 0.7146 across the folds. If this was the final model this would be a weak classifier. By tuning this model with varying parameters, performance will hopefully incease.  increase. 

### Model feature exploration

In the code cell below, explore different features of XGBoost and how they work (e.g. scale_pos_weight, max_depth, learning_rate).
Use stratified k-fold cross or repeated stratified k-fold cross validation with your model building. 
You should explore at least 3 different features of XGBoost.
Identify the model that performs best.

In [18]:
# hyperparameter tuning of xgboost with stratified k-fold cross validation on at least 3 different hyperparameters
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5],
    'learning_rate': [0.01, 0.1]
}
xgb = XGBClassifier(enable_categorical=True, random_state=222)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)

# Use GridSearchCV to tune hyperparameters with stratified k-fold
grid_search = GridSearchCV(estimator=xgb, param_grid=param_grid, cv=skf, scoring='f1', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best model and its parameters
best_xgb = grid_search.best_estimator_
print("Best hyperparameters:", grid_search.best_params_)
print(f"Best cross-validated F1 score: {grid_search.best_score_:.4f}")

# Evaluate on test set
y_pred = best_xgb.predict(X_test)
from sklearn.metrics import f1_score
test_f1 = f1_score(y_test, y_pred, average='weighted')
print(f"Test F1 score: {test_f1:.4f}")


Best hyperparameters: {'learning_rate': 0.1, 'max_depth': 5, 'n_estimators': 200}
Best cross-validated F1 score: 0.7165
Test F1 score: 0.8693


### Tuning with GridSearchCV

Use the code cell below to set up your parameter grid and run GridSearchCV with your preferred model from above. You should tune 4-5 hyperparameters utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [ ]:
# Expanded parameter grid for GridSearchCV - tuning 5 hyperparameters
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 2, 3]
}

# Initialize XGBoost classifier
xgb = XGBClassifier(enable_categorical=True, random_state=222)

# Set up StratifiedKFold for cross-validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=222)

# Perform GridSearchCV
grid_search = GridSearchCV(
    estimator=xgb, 
    param_grid=param_grid, 
    cv=skf, 
    scoring='f1', 
    n_jobs=-1,  # Use all available cores
    verbose=1
)

# Fit on training data
grid_search.fit(X_train, y_train)

# Print best parameters and CV score
print("Best hyperparameters from GridSearchCV:")
print(grid_search.best_params_)
print(f"\nBest cross-validated F1 score: {grid_search.best_score_:.4f}")

# Train final model with best parameters
best_xgb = grid_search.best_estimator_

# Make predictions on test set
y_pred = best_xgb.predict(X_test)

# Evaluate performance
from sklearn.metrics import classification_report, confusion_matrix, f1_score, accuracy_score

print(f"\nTest Set Performance:")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 Score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print(f"F1 Score (macro): {f1_score(y_test, y_pred, average='macro'):.4f}")

print(f"\nClassification Report:")
print(classification_report(y_test, y_pred))

print(f"\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Fitting 5 folds for each of 729 candidates, totalling 3645 fits
Best hyperparameters from GridSearchCV:
{'colsample_bytree': 0.9, 'learning_rate': 0.2, 'max_depth': 5, 'n_estimators': 100, 'scale_pos_weight': 2, 'subsample': 0.9}

Best cross-validated F1 score: 0.7320

Test Set Performance:
Accuracy: 0.8606
F1 Score (weighted): 0.8640
F1 Score (macro): 0.8226

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      6803
           1       0.69      0.80      0.74      2242

    accuracy                           0.86      9045
   macro avg       0.81      0.84      0.82      9045
weighted avg       0.87      0.86      0.86      9045


Confusion Matrix:
[[5985  818]
 [ 443 1799]]


### Tuning with RandomizedSearchCV

Using the code cell below as a starting point, tune your preferred model from above. Tune the same 4-5 hyperparameters from above utilizing cross validation. Train a final model using the best hyperparameters and report your model performance.

In [20]:
# tuning xgboost classifier with RandomizedSearchCV (tune more parameters than shown here)
param_dist = {
        'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.8, 0.9, 1.0],
    'colsample_bytree': [0.8, 0.9, 1.0],
    'scale_pos_weight': [1, 2, 3]
}

# replace this placeholder model with your preferred model from above

xgb_random = RandomizedSearchCV(XGBClassifier(random_state=222, enable_categorical=True),
                                param_distributions=param_dist, n_iter=20, cv=skf, scoring='f1', random_state=222)
xgb_random.fit(X_train, y_train)
print(f'Best parameters from RandomizedSearchCV: {xgb_random.best_params_}')
print(f'Best F1 score from RandomizedSearchCV: {xgb_random.best_score_}')   

# build your preferred model from above using best parameters from your RandomizedSearchCV
# and evaluate on the test set

xgb_random_best = XGBClassifier(random_state=222, scale_pos_weight=xgb_random.best_params_['scale_pos_weight'], 
                                max_depth=xgb_random.best_params_['max_depth'], 
                                learning_rate=xgb_random.best_params_['learning_rate'],
                                n_estimators=xgb_random.best_params_['n_estimators'],
                                subsample=xgb_random.best_params_['subsample'],
                                colsample_bytree=xgb_random.best_params_['colsample_bytree'], 
                                enable_categorical=True)
                                
xgb_random_best.fit(X_train, y_train)
y_pred_random = xgb_random_best.predict(X_test)
print(f'Classification report for RandomizedSearchCV-tuned model:\n{classification_report(y_test, y_pred_random)}')


Best parameters from RandomizedSearchCV: {'subsample': 0.9, 'scale_pos_weight': 2, 'n_estimators': 100, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 1.0}
Best F1 score from RandomizedSearchCV: 0.7316442728790649
Classification report for RandomizedSearchCV-tuned model:
              precision    recall  f1-score   support

           0       0.93      0.88      0.90      6803
           1       0.68      0.80      0.74      2242

    accuracy                           0.86      9045
   macro avg       0.80      0.84      0.82      9045
weighted avg       0.87      0.86      0.86      9045



### Tuning with Optuna

Using the code cell below as a starting point, tune your preferred model from above. You should tune the same 4-5 parameters as above using cross validation. Train a final model using the best hyperparameters and report your model performance.

In [23]:
# tuning with Optuna (tune more parameters than shown here)
def objective(trial):
    n_estimators = trial.suggest_categorical('n_estimators', [100, 200, 300])
    scale_pos_weight = trial.suggest_categorical('scale_pos_weight', [1.0, 2.0, 3.0])
    max_depth = trial.suggest_categorical('max_depth', [3, 5, 7])
    learning_rate = trial.suggest_categorical('learning_rate', [0.01, 0.1, 0.2])
    subsample = trial.suggest_categorical('subsample', [0.8, 0.9, 1.0])
    colsample_bytree = trial.suggest_categorical('colsample_bytree', [0.8, 0.9, 1.0])
    
    # replace this placeholder model with your preferred model from above
    
    xgb_optuna = XGBClassifier(random_state=222, n_estimators=n_estimators, scale_pos_weight=scale_pos_weight, 
                               max_depth=max_depth,  learning_rate=learning_rate, subsample=subsample, colsample_bytree=colsample_bytree, enable_categorical=True)
    
    cv_scores = cross_val_score(xgb_optuna, X, y, cv=skf, scoring='f1')
    return cv_scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20, show_progress_bar=True)

print(f'Best parameters from Optuna: {study.best_params}')
print(f'Best F1 score from Optuna: {study.best_value}')

# build preferred model from above with best parameters from Optuna and evaluate on the test set
xgb_optuna_best = XGBClassifier(random_state=222, n_estimators=study.best_params['n_estimators'], scale_pos_weight=study.best_params['scale_pos_weight'], 
                                  max_depth=study.best_params['max_depth'], 
                                  learning_rate=study.best_params['learning_rate'], 
                                  subsample=study.best_params['subsample'], 
                                  colsample_bytree=study.best_params['colsample_bytree'], 
                                  enable_categorical=True)

xgb_optuna_best.fit(X_train, y_train)
y_pred_optuna = xgb_optuna_best.predict(X_test)
print(f'Classification report for Optuna-tuned model:\n{classification_report(y_test, y_pred_optuna)}')


[I 2026-04-15 13:01:17,507] A new study created in memory with name: no-name-8e4b37ff-d3d2-4860-87bc-d1360ba3d2e8


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-04-15 13:01:19,820] Trial 0 finished with value: 0.7283569206816403 and parameters: {'n_estimators': 200, 'scale_pos_weight': 2.0, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 1.0, 'colsample_bytree': 0.8}. Best is trial 0 with value: 0.7283569206816403.
[I 2026-04-15 13:01:22,402] Trial 1 finished with value: 0.7159134719702333 and parameters: {'n_estimators': 200, 'scale_pos_weight': 3.0, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.8, 'colsample_bytree': 1.0}. Best is trial 0 with value: 0.7283569206816403.
[I 2026-04-15 13:01:25,662] Trial 2 finished with value: 0.6889392977641224 and parameters: {'n_estimators': 200, 'scale_pos_weight': 3.0, 'max_depth': 5, 'learning_rate': 0.01, 'subsample': 1.0, 'colsample_bytree': 0.9}. Best is trial 0 with value: 0.7283569206816403.
[I 2026-04-15 13:01:27,316] Trial 3 finished with value: 0.7069949286663337 and parameters: {'n_estimators': 100, 'scale_pos_weight': 3.0, 'max_depth': 3, 'learning_rate': 0.1, 'subsample': 0.

### Tuning results

In the markdown cell below describe your experience tuning with the different methods. Which produced the best results? Which do you prefer?


All of the models produced remarkably similar results. Optuna produced a slighlty higher F1 score (0.7329) than GridSearchCV (0.7320), and RandomizedSearchCV (0.7316). The major benefit of Optuna is how fast it was, executing in 1m30s compared to the 12m46s of GridSearch. 

Due to faster execution and improved results, I prefer Optuna. The downside is having to learn the different syntax compared to GridSearchCV and RandomizedSearchCV, but speed and higher predictive ability is well worth it.